In [1]:
# Prompt Chaining : involves multiple LLM calls
#Problem Statement : Based on title provided:
# 1. generate outline of blog
# 2. based on title and outline, generate content of blog
# 3. Evaluate the content, such that how much it is matching with outline generated.

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint 
from dotenv import load_dotenv

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
load_dotenv()

True

In [4]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct"  # model name
)

model = ChatHuggingFace(llm=llm)  # Create a LangChain chat model using the Hugging Face endpoint.


In [5]:
# create state class
class PromptChainState(TypedDict):
    title : str
    outline : str
    content : str
    score : int


In [6]:
def generate_outline(state:PromptChainState)->PromptChainState: 
    #fetch the title
    title = state['title']
    #convert it to prompt
    prompt = f"Generate a detailed outline for a blog on topic : {title}"
    #give input to model
    outline = model.invoke(prompt).content
    #update state
    state['outline'] = outline
    return state
    

In [7]:
def generate_content(state:PromptChainState)->PromptChainState: 
    #fetch the title and outline
    title = state['title']
    outline = state['outline']
    #convert it to prompt
    prompt = f"write a detailed blog on topic : {title}, using following outline:\n {outline}"
    #give input to model
    content = model.invoke(prompt).content
    #update state
    state['content'] = content
    return state

In [8]:
def evaluate_blog(state:PromptChainState)->PromptChainState:
    #Fetch title, outline,content
    title = state['title']
    outline = state['outline']
    content = state['content']
    #convert it to prompt
    prompt = f"Evaluate the blog provided below, based on how much it matches the given outline and rate it in range of 1(worst)-5(best) in integer only. Title of blog is : {title}. outline is : {outline}. content is : {content}."
    #give input to model
    rating = model.invoke(prompt).content
    #update state
    state['score'] = rating
    return state

In [9]:
graph = StateGraph(PromptChainState)
graph.add_node('generate_outline',generate_outline)
graph.add_node('generate_content',generate_content)
graph.add_node('evaluate_blog',evaluate_blog)
graph.add_edge(START,'generate_outline')
graph.add_edge('generate_outline','generate_content')
graph.add_edge('generate_content','evaluate_blog')
graph.add_edge('evaluate_blog',END)

workflow = graph.compile()

In [22]:
initial_state = {'title':'Journey of Virat Kohli from 2011 to 2023 in ICC tournaments'}
final_state = workflow.invoke(initial_state)

In [23]:
print(final_state)

{'title': 'Journey of Virat Kohli from 2011 to 2023 in ICC tournaments', 'outline': 'Here is a detailed outline for a blog on the journey of Virat Kohli in ICC tournaments from 2011 to 2023:\n\n**Title:** "The Rise of a Legend: Virat Kohli\'s Journey in ICC Tournaments from 2011 to 2023"\n\n**I. Introduction**\n\n* Brief overview of Virat Kohli\'s career and his achievements in ICC tournaments\n* Importance of analyzing his journey in detail\n* Thesis statement: Virat Kohli\'s transformation from a young and talented player to a consistent and dominant force in ICC tournaments is a testament to his hard work, dedication, and passion for the game.\n\n**II. Early Days (2011-2012)**\n\n* Virat Kohli\'s debut in the ICC World Cup 2011 and his initial struggles\n* His performance in the 2011-12 season, including his first international century\n* Transition from a domestic star to an international player\n\n**III. Emergence as a Middle-Order Batsman (2013-2014)**\n\n* Virat Kohli\'s breakth

In [24]:
print(final_state['outline'])

Here is a detailed outline for a blog on the journey of Virat Kohli in ICC tournaments from 2011 to 2023:

**Title:** "The Rise of a Legend: Virat Kohli's Journey in ICC Tournaments from 2011 to 2023"

**I. Introduction**

* Brief overview of Virat Kohli's career and his achievements in ICC tournaments
* Importance of analyzing his journey in detail
* Thesis statement: Virat Kohli's transformation from a young and talented player to a consistent and dominant force in ICC tournaments is a testament to his hard work, dedication, and passion for the game.

**II. Early Days (2011-2012)**

* Virat Kohli's debut in the ICC World Cup 2011 and his initial struggles
* His performance in the 2011-12 season, including his first international century
* Transition from a domestic star to an international player

**III. Emergence as a Middle-Order Batsman (2013-2014)**

* Virat Kohli's breakthrough series in the 2013 Champions Trophy, where he scored 281 runs at an average of 140.50
* His performanc

In [25]:
print(final_state['content'])

**The Rise of a Legend: Virat Kohli's Journey in ICC Tournaments from 2011 to 2023**

Virat Kohli, one of the most successful batsmen in the history of cricket, has been a dominant force in the ICC tournaments since his debut in 2011. His journey from a young and talented player to a consistent and dominant force in the game is a testament to his hard work, dedication, and passion for cricket. In this blog, we will take a detailed look at Virat Kohli's journey in the ICC tournaments from 2011 to 2023, highlighting his growth, achievements, and challenges.

**II. Early Days (2011-2012)**

Virat Kohli made his debut in the ICC World Cup 2011, but his early days in international cricket were marked by struggles. In the tournament, he scored 135 runs at an average of 27, which was not enough to make a significant impact. However, he did show glimpses of his talent, particularly in the match against Pakistan, where he scored 78 runs.

In the 2011-12 season, Kohli's performance improved sign

In [26]:
print(final_state['score'])

Here's a rating of the blog post based on the given outline:

**I. Introduction**: 4/5 (The blog post has a clear introduction that sets the tone for the rest of the article, but it doesn't fully match the outline. The thesis statement is also not explicitly stated.)

**II. Early Days (2011-2012)**: 4/5 (The section provides a good overview of Kohli's early days in international cricket, but it doesn't delve as deeply into the details as the outline suggests.)

**III. Emergence as a Middle-Order Batsman (2013-2014)**: 4/5 (The section is well-written and provides a good analysis of Kohli's growth as a batsman, but it doesn't fully match the outline in terms of the specific details and statistics.)

**IV. The Rise of a Great (2015-2016)**: 4/5 (The section is well-written and provides a good analysis of Kohli's exceptional form in the 2015 ICC Cricket World Cup, but it doesn't fully match the outline in terms of the specific details and statistics.)

**V. The Golden Period (2017-2019)**